In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("train_IPL.csv")
test_data = pd.read_csv("public_lb_matches.csv")

data = data.fillna(0)
test_data = test_data.fillna(0)

if "Date" in data.columns:
    data["Date"] = pd.to_datetime(data["Date"])
    data["year"] = data["Date"].dt.year
    data["month"] = data["Date"].dt.month
    data["day"] = data["Date"].dt.day
    data = data.drop("Date", axis=1)

from sklearn.preprocessing import LabelEncoder

categorical_columns = data.select_dtypes(include=["object"]).columns

for col in categorical_columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

test_categorical = test_data.select_dtypes(include=["object"]).columns

for col in test_categorical:
    le = LabelEncoder()
    test_data[col] = le.fit_transform(test_data[col].astype(str))

leakage_columns = [
    "match_won_by",
    "match_winner",
    "Winner",
    "winner",
    "winning_team",
    "result",
    "result_margin",
    "won_by",
    "result_type",
    "Innings Runs",
    "Innings Wickets",
    "Target Score",
    "Runs to Get",
    "Balls Remaining",
    "Total Batter Runs",
    "Total Non Striker Runs",
    "Batter Balls Faced",
    "Non Striker Balls Faced",
    "Player Out Runs",
    "Player Out Balls Faced",
    "Bowler Runs Conceded"
]
print(data.columns)

X = data.drop(
    columns=[col for col in leakage_columns if col in data.columns],
    errors="ignore"
)


y = data["match_won_by"]

X = X.select_dtypes(include=["number"])
remove_cols = [
    "Innings Runs",
    "Innings Wickets",
    "Target Score",
    "Runs to Get",
    "Balls Remaining",
    "Total Batter Runs",
    "Total Non Striker Runs",
    "Batter Balls Faced",
    "Non Striker Balls Faced",
    "Player Out Runs",
    "Player Out Balls Faced",
    "Bowler Runs Conceded",
    "match_won_by",
    "Batter Runs",
    "Extra Runs",
    "Runs From Ball",
    "Wicket",
    "Valid Ball"
]
X = X.drop(columns=[col for col in remove_cols if col in X.columns], errors="ignore")
test_data = test_data.select_dtypes(include=["number"])

test_data = test_data.reindex(columns=X.columns, fill_value=0)

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(x_train, y_train)

y_pred = model.predict(x_test)

from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

test_probs = model.predict_proba(test_data)

submission = pd.DataFrame(
    test_probs,
    columns=model.classes_
)

submission.insert(0, "match_id", test_data.index)

submission.to_csv("submission.csv", index=False)

print(submission.head())

C:\Users\sidak\AppData\Local\Temp\ipykernel_44900\155302972.py:4: DtypeWarning: Columns (35,36) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("train_IPL.csv")


Index(['Match ID', 'Venue', 'Bat First', 'Bat Second', 'Innings', 'Over',
       'Ball', 'Batter', 'Non Striker', 'Bowler', 'Batter Runs', 'Extra Runs',
       'Runs From Ball', 'Ball Rebowled', 'Extra Type', 'Wicket',
       'Dismissal Method', 'Player Out', 'Innings Runs', 'Innings Wickets',
       'Target Score', 'Runs to Get', 'Balls Remaining', 'Total Batter Runs',
       'Total Non Striker Runs', 'Batter Balls Faced',
       'Non Striker Balls Faced', 'Player Out Runs', 'Player Out Balls Faced',
       'Bowler Runs Conceded', 'Valid Ball', 'toss_winner', 'toss_decision',
       'city', 'result_type', 'season', 'match_won_by', 'year', 'month',
       'day'],
      dtype='object')
Accuracy: 1.0
   match_id     0     1     2     3     4     5     6     7     8  ...    10  \
0         0  0.27  0.10  0.01  0.06  0.00  0.02  0.07  0.04  0.09  ...  0.11   
1         1  0.29  0.04  0.03  0.08  0.00  0.04  0.10  0.00  0.05  ...  0.07   
2         2  0.31  0.11  0.06  0.09  0.00  0.00  0.0